# DuckDB & Parquet - Bloom Filter

Predicate pushdown inside a Parquet file is based on columns statistics (stored in metadata), but also on [Bloom filters](https://en.wikipedia.org/wiki/Bloom_filter) (fast exclusion of row groups):

- <https://duckdb.org/docs/current/data/parquet/metadata#bloom-filters>.

⚠️ This notebook uses the Parquet file `../data/sample.parquet` generated by the notebook `notebook/07-duckdb_parquet-row_groups.ipynb`.

In [1]:
%config SqlMagic.displaylimit = 100
%config SqlMagic.autopandas = False

In [2]:
%%capture
%load_ext sql

In [3]:
# https://duckdb.org/docs/lts/guides/python/jupyter
%sql duckdb:///:memory: --alias duckdb

Connecting to 'duckdb'

Inspect the row groups **#7** and **#8** for the field `code`:

In [4]:
%%sql

SELECT
    row_group_id,
    path_in_schema,
    stats_min,
    stats_max,
    stats_distinct_count,
    bloom_filter_offset,
    bloom_filter_length,
FROM parquet_metadata('../data/sample.parquet')
WHERE path_in_schema = 'code' AND row_group_id IN (7, 8);

Running query in 'duckdb'

row_group_id,path_in_schema,stats_min,stats_max,stats_distinct_count,bloom_filter_offset,bloom_filter_length
7,code,l,m,2,417774,47
8,code,m,o,3,417821,47


Check the Bloom filter.

All row groups will be **excluded** but:

- **#7** (`stats_min="l"`, `statis_max="m"`).
- **#8** (`stats_min="m"`, `statis_max="o"`).

They both contain the letter **"m"**.

In [5]:
%%sql

FROM parquet_bloom_probe('../data/sample.parquet', 'code', 'm');

Running query in 'duckdb'

file_name,row_group_id,bloom_filter_excludes
../data/sample.parquet,0,True
../data/sample.parquet,1,True
../data/sample.parquet,2,True
../data/sample.parquet,3,True
../data/sample.parquet,4,True
../data/sample.parquet,5,True
../data/sample.parquet,6,True
../data/sample.parquet,7,False
../data/sample.parquet,8,False
../data/sample.parquet,9,True


The Bloom filter will help **selective reading**, even if the column's data is **randomly shuffled** (i.e. `stats_min` and `stats_max` can't be used).

💡 The bloom filter generation (controlled by the option `WRITE_BLOOM_FILTER`, default to `true`) may not apply, depending on the size of the dictionary (option `DICTIONARY_SIZE_LIMIT`).  
See the DuckDB [parquet options](https://duckdb.org/docs/current/sql/statements/copy#parquet-options), [Parquet Bloom Filters in DuckDB](https://duckdb.org/2025/03/07/parquet-bloom-filters-in-duckdb) and [Bloom Filter Support in Parquet Reader/Writer](https://github.com/duckdb/duckdb/pull/14597) for more details.